In [79]:
import pandas as pd
from Bio import Phylo

# --- Input files ---
host_tree_file = "host_taxonomy_tree_cow.newick"
virus_tree_file = "virus_tree_cow.newick"
interaction_tsv = "cow_dataset_hic_gtdb.tsv"
output_csv = "interaction_matrix_cow.csv"

# --- Load the interaction table ---
df = pd.read_csv(interaction_tsv, sep="\t")

# --- Extract phages and hosts from the TSV ---
phages = df['Phage'].unique()
hosts = df['MAGs'].unique()

# --- Parse Newick trees using Bio.Phylo ---
host_tree = Phylo.read(host_tree_file, "newick")
virus_tree = Phylo.read(virus_tree_file, "newick")

# Get ordered leaf names
ordered_hosts = [term.name for term in host_tree.get_terminals() if term.name in hosts]
ordered_phages = [term.name for term in virus_tree.get_terminals() if term.name in phages]

# --- Initialize empty interaction matrix ---
interaction_matrix = pd.DataFrame(0, index=ordered_phages, columns=ordered_hosts)

# --- Fill the matrix with interactions ---
for _, row in df.iterrows():
    phage = row['Phage']
    host = row['MAGs']
    if phage in interaction_matrix.index and host in interaction_matrix.columns:
        interaction_matrix.loc[phage, host] = 1

# --- Save to CSV ---
interaction_matrix.to_csv(output_csv)
print(f"Interaction matrix saved to {output_csv}")


Interaction matrix saved to interaction_matrix_cow.csv


In [80]:
import argparse
import os
import numpy as np
from Bio import Phylo
import sys
import os
import copy
import math
import json
import pandas as pd
import matplotlib.pyplot as plt

# Add parent directory to sys.path
sys.path.append(os.path.abspath(".."))
import importlib
# importlib.reload(sys.modules['simulation_utils'])
from simulation_utils import *

In [120]:
def read_interaction_matrix(csv_file, host_tree_file, virus_tree_file):
    """
    Read host-virus interaction matrix CSV and reorder according to tree leaves.
    Treat viruses as parasites.
    
    Returns:
        cell_state: dict with keys (virus, host) -> 0/1
        host_tree: Bio.Phylo tree object
        virus_tree: Bio.Phylo tree object
    """

    # Load trees
    host_tree = Phylo.read(host_tree_file, "newick")
    virus_tree = Phylo.read(virus_tree_file, "newick")

    # Get leaf order
    host_order = [t.name for t in host_tree.get_terminals()]
    virus_order = [t.name for t in virus_tree.get_terminals()]

    print(f"Number of hosts in tree: {len(host_order)}")
    print(f"Number of viruses in tree: {len(virus_order)}")

    # Load interaction matrix CSV
    df = pd.read_csv(csv_file, index_col=0)  # assuming rows=viruses, columns=hosts
    # Ensure names match the expected tree leaf names
    df = df.loc[df.index.intersection(virus_order), df.columns.intersection(host_order)]

    # Reorder according to tree leaves
    df = df.reindex(index=virus_order, columns=host_order, fill_value=0)

    # Build cell_state dictionary
    cell_state = {}
    for virus in virus_order:
        for host in host_order:
            cell_state[(virus, host)] = int(df.at[virus, host])

    return cell_state, host_tree, virus_tree







outdir = "experiments"
association_csv = "interaction_matrix_cow.csv"
host_tree = "host_taxonomy_tree_cow.newick"
virus_tree = "virus_tree_cow.newick"


os.makedirs(outdir, exist_ok=True)

# Read associations + trees
cell_state, base_host_tree, base_par_tree = read_interaction_matrix(
    association_csv, host_tree, virus_tree
)

base_host_tree = rescale_tree(base_host_tree)
base_par_tree = rescale_tree(base_par_tree)

# print (base_host_tree)
# print (base_par_tree)

host_leaves = [t.name for t in base_host_tree.get_terminals()]
par_leaves = [t.name for t in base_par_tree.get_terminals()]

out = {
    "host_trees": [copy.deepcopy(base_host_tree) for _ in range(len(par_leaves))],
    "par_trees": [copy.deepcopy(base_par_tree) for _ in range(len(host_leaves))],
    "cell_state": cell_state,
    "host_leaves": host_leaves,
    "par_leaves": par_leaves
}

# Build the matrix
mat, parasites, hosts = get_interaction_matrix(out)

# Save the matrix visualization
# plot_matrix(mat, parasites, hosts, filename=os.path.join(outdir, "original_matrix_cow.svg"))

Number of hosts in tree: 66
Number of viruses in tree: 66


In [136]:
import numpy as np
import matplotlib.pyplot as plt

import pandas as pd
import numpy as np

def parse_phist_costs(csv_path, hosts, parasites):
    """
    Build a flip cost matrix using k-mer CSV file, mapping parasites (rows) × hosts (columns).
    Each cell = k-mer count from CSV, or 0 if missing.
    """
    with open(csv_path) as f:
        lines = [line.strip() for line in f if line.strip()]
    
    # Extract phage names from the header (3rd column onwards)
    header_line = next(l for l in lines if l.startswith("kmer-length"))
    header_parts = header_line.split(",")
    phage_names = [p.replace(".fa", "") for p in header_parts[2:] if p.startswith("k141_")]
    
    # Map phage index (1-based) → phage name
    phage_index = {i+1: name for i, name in enumerate(phage_names)}
    
    # Prepare empty cost matrix: rows = parasites, cols = hosts
    cost_matrix = np.zeros((len(parasites), len(hosts)), dtype=int)
    
    for line in lines:
        if not line.startswith("bin."):
            continue
        
        parts = line.split(",")
        host_name = parts[0].replace(".fa", "")
        if host_name not in hosts:
            continue
        
        host_idx = hosts.index(host_name)
        
        # Iterate over all kmer pairs like 4:10
        for token in parts[2:]:
            if ":" not in token:
                continue
            try:
                phage_num, value = map(int, token.split(":"))
            except ValueError:
                continue
            
            if phage_num not in phage_index:
                continue
            phage_name = phage_index[phage_num]
            
            if phage_name not in parasites:
                continue
            
            phage_idx = parasites.index(phage_name)
            cost_matrix[phage_idx, host_idx] = value

    # row_sums = cost_matrix.sum(axis=1, keepdims=True)
    #     # Avoid division by zero
    # row_sums[row_sums == 0] = 1
    # cost_matrix = cost_matrix / row_sums
            

    # Return DataFrame (rows = viruses, columns = hosts)

    return cost_matrix

def parse_pblks_costs(csv_path, hosts, parasites):
    """
    Parse a binary virus–host matrix and assign weighted costs.
    For each row (virus):
      - If entry == 1: cost = column_index / (number of 1s in that row)
      - If entry == 0: cost = 1
    Column priority = left to right (1-based index).
    """
    # Load CSV
    df = pd.read_csv(csv_path, index_col=0)

    # Clean up names
    df.index = df.index.str.replace(".fa", "", regex=False)
    df.columns = df.columns.str.replace(".fa", "", regex=False)

    # Keep only intersecting rows/cols before weighting
    df = df.loc[df.index.intersection(parasites), df.columns.intersection(hosts)]

    # Convert to numeric (in case CSV parsed as str)
    df = df.apply(pd.to_numeric, errors="coerce").fillna(0).astype(int)

    # Initialize cost matrix (float)
    cost_df = pd.DataFrame(2.0, index=df.index, columns=df.columns)

    # Compute weighted costs
    for i, row in df.iterrows():
        ones = row.values == 1
        num_ones = ones.sum()
        if num_ones == 0:
            continue  # all costs remain 1
        col_indices = np.arange(1, len(row) + 1)  # 1-based column indices
        weights = col_indices / len(df.columns)
        cost_df.loc[i, ones] = weights[ones]

    # Reindex to match provided order and fill missing with 1
    cost_df = cost_df.reindex(index=parasites, columns=hosts, fill_value=2.0)

    # Return as numpy float array
    return cost_df.to_numpy(dtype=float)


def parse_wish_costs(matrix_path, hosts, parasites):
    """
    Parse a virus–host cost matrix and assign rank-based costs
    derived from log-likelihood scores.

    For each phage (column):
      - Rank hosts by descending score (higher = better)
      - Assign cost = rank / total_hosts (so smaller = better)

    Parameters
    ----------
    matrix_path : str
        Path to the .matrix file (TSV: rows = hosts, columns = parasites)
    hosts : list[str]
        List of host (MAG/bin) names to retain and order.
    parasites : list[str]
        List of parasite (phage) names to retain and order.

    Returns
    -------
    cost_matrix : np.ndarray
        2D array (len(parasites) × len(hosts)), ordered as given.
    """

    # Load tab-separated matrix (first column = row names)
    df = pd.read_csv(matrix_path, sep='\t', index_col=0)

    # Clean up names
    df.index = df.index.str.replace(".fa", "", regex=False)
    df.columns = df.columns.str.replace(".fa", "", regex=False)

    # Filter to intersection first
    df = df.loc[df.index.intersection(hosts), df.columns.intersection(parasites)]

    # Reindex to given order (fill missing with NaN)
    df = df.reindex(index=hosts, columns=parasites).astype(float)

    # Initialize cost DataFrame
    cost_df = pd.DataFrame(index=hosts, columns=parasites, dtype=float)

    # Assign rank-based costs for each parasite (phage)
    for phage in df.columns:
        scores = df[phage]
        n_hosts = scores.notna().sum()
        if n_hosts == 0:
            cost_df[phage] = np.nan
            continue

        # Rank hosts: high score → low rank (1 = best)
        ranked_hosts = scores.rank(ascending=False, method='first')

        # Cost = rank / total_hosts (so 1st = 1/n, last = 1)
        cost_df[phage] = ranked_hosts / n_hosts

    # Transpose → rows = parasites, columns = hosts
    cost_df = cost_df.T.reindex(index=parasites, columns=hosts, fill_value=1.0)

    # Return as NumPy array
    return cost_df.to_numpy(dtype=float)


# flip_cost_matrix = 1/parse_phist_costs("../../HostPredictionReview/Benchmark/Task2-MetaHiC/Results/phist/common_kmers_water.csv", hosts, parasites)
# flip_cost_matrix = parse_pblks_costs("../../HostPredictionReview/Benchmark/Task2-MetaHiC/Results/pblks/water.csv", hosts, parasites)
flip_cost_matrix = parse_wish_costs("../../HostPredictionReview/Benchmark/Task2-MetaHiC/Results/wish/cow.matrix", hosts, parasites)
# normalize to [0,1] in each row

# make the inf entries 1
# flip_cost_matrix[np.isinf(flip_cost_matrix)] = 1
flip_cost_matrix=(flip_cost_matrix)*100
# flip_cost_df.to_csv("flip_cost_matrix.csv")


# def compute_flipping_cost_matrix(mat):
#     """
#     Compute a flipping-cost matrix for all cells in the binary matrix.
#     Each cell's cost is proportional to number of nearby 1's, weighted by 1/distance^2.
#     """
#     n_rows, n_cols = mat.shape
#     cost_mat = np.zeros_like(mat, dtype=float)

#     # Pad matrix to simplify boundary handling
#     pad = max(n_rows, n_cols)
#     padded = np.pad(mat, pad_width=pad, mode='constant', constant_values=0)
#     offset = pad

#     for i in range(n_rows):
#         for j in range(n_cols):
#             total_score = 0.0
#             max_k = max(n_rows, n_cols) // 2

#             r_p, c_p = i + offset, j + offset

#             for k in range(1, max_k + 1):
#                 top, bottom = r_p - k, r_p + k
#                 left, right = c_p - k, c_p + k

#                 # extract boundary cells
#                 boundary = np.concatenate([
#                     padded[top, left:right+1],
#                     padded[bottom, left:right+1],
#                     padded[top+1:bottom, left],
#                     padded[top+1:bottom, right]
#                 ])

#                 num_ones = np.sum(boundary == 1)
#                 total_score += num_ones / (k ** 2)

#             cost_mat[i, j] = total_score + mat[i, j]

#     # Optional: normalize for visualization
#     if cost_mat.max() > 0:
#         cost_mat = cost_mat / cost_mat.max()

#     return cost_mat


# def plot_heatmap(matrix, title="Flipping Cost Heatmap"):
#     """Simple matrix heatmap plot."""
#     plt.figure(figsize=(6, 6))
#     plt.imshow(matrix, cmap='hot', interpolation='nearest')
#     plt.colorbar(label='Flipping Cost')
#     plt.title(title)
#     plt.xlabel("Hosts")
#     plt.ylabel("Phages")
#     plt.show()

# flipping_cost_matrix = compute_flipping_cost_matrix(mat)
# plot_heatmap(flipping_cost_matrix, title="Flipping Cost Heatmap")

In [137]:
flip_cost_matrix[0]

array([ 34.84848485,  43.93939394,  42.42424242,  12.12121212,
        51.51515152,  84.84848485,  77.27272727,  71.21212121,
        54.54545455,  66.66666667,  60.60606061,  48.48484848,
        72.72727273,  86.36363636,  74.24242424,  92.42424242,
        25.75757576,  59.09090909,  81.81818182,  93.93939394,
       100.        ,  96.96969697,  98.48484848,  40.90909091,
        50.        ,  90.90909091,  45.45454545,  39.39393939,
        30.3030303 ,  63.63636364,  27.27272727,   3.03030303,
         6.06060606,  24.24242424,  31.81818182,  28.78787879,
        37.87878788,  68.18181818,  83.33333333,  36.36363636,
         1.51515152,  57.57575758,  78.78787879,  62.12121212,
        22.72727273,   9.09090909,   7.57575758,  18.18181818,
         4.54545455,  65.15151515,  16.66666667,  33.33333333,
        13.63636364,  21.21212121,  19.6969697 ,  95.45454545,
        89.39393939,  46.96969697,  56.06060606,  87.87878788,
        75.75757576,  10.60606061,  69.6969697 ,  80.30

In [123]:
out_original = out.copy()  # Keep original for each run

In [23]:
out=out_original.copy()

In [131]:
def matrix_builder(out, r01_h, r10_h, r01_p, r10_p):
    host_W_matrices = []
    for t in out["host_trees"]:
        W, nodes = build_weight_matrix(t, r01_h, r10_h, scale=100.0)
        # scale W
        # W[W!=0] =  np.inf
        W = W*100
        host_W_matrices.append((W, nodes))

    # Build weight matrices for each host’s parasite tree
    par_W_matrices = []
    for t in out["par_trees"]:
        W, nodes = build_weight_matrix(t, r01_p, r10_p, scale=100.0)
        # scale W
        W = W*100
        par_W_matrices.append((W, nodes))
    # Flip cost matrix
    # flip_cost_matrix = build_flip_cost_matrix(len(out["par_leaves"]), len(out["host_leaves"]), cost=100.0)
    # flip_cost_matrix=-np.log(flipping_cost_matrix)
    return host_W_matrices, par_W_matrices
out_corrupt = out_original.copy()
hidden_cells = []
# for virus in parasites:
#     for host in hosts:
#         if out_corrupt["cell_state"][(virus, host)] == 1 and host in betacov_true_hosts_id:
#             hidden_cells.append((virus, host))
#             out_corrupt["cell_state"][(virus, host)] = 0
seed = 42
r01_p, r10_p, r01_h, r10_h = 0.5,0.5,0.5,0.5
host_W_matrices, par_W_matrices = matrix_builder(out_corrupt, r01_h, r10_h, r01_p, r10_p)

In [132]:
par_W_matrices[0][0]

array([[ 0.        , 50.00311873, 51.51068698, ...,  0.        ,
         0.        ,  0.        ],
       [50.00311873,  0.        ,  0.        , ...,  0.        ,
         0.        ,  0.        ],
       [51.51068698,  0.        ,  0.        , ...,  0.        ,
         0.        ,  0.        ],
       ...,
       [ 0.        ,  0.        ,  0.        , ...,  0.        ,
         0.        ,  0.        ],
       [ 0.        ,  0.        ,  0.        , ...,  0.        ,
         0.        ,  0.        ],
       [ 0.        ,  0.        ,  0.        , ...,  0.        ,
         0.        ,  0.        ]], shape=(130, 130))

In [133]:
def flip_specific_cells(mat, parasites, hosts, flip_targets):
    """
    Flip given cells (1->0) based on Phage-HOST pairs.
    Also set flip cost rules:
      - Flipped cells -> cost = 0.5
      - All other phage rows (not in flip targets) -> inf for all hosts
    
    Args:
        mat: numpy array, original matrix
        parasites: list of row labels (Phages)
        hosts: list of column labels (MAGs)
        flip_targets: list of tuples (Phage, MAGs) to flip
        flip_cost_matrix: numpy array (same shape as mat) to modify
    
    Returns:
        mat_corrupt: numpy array with specified cells flipped
        flipped_indices: list of (i,j) indices that were flipped
        flip_cost_matrix: modified cost matrix
    """
    mat_corrupt = mat.copy()
    flipped_indices = []

    parasite_idx = {p: i for i, p in enumerate(parasites)}
    host_idx = {h: j for j, h in enumerate(hosts)}

    # Track which phages are in flip targets
    targeted_phages = set()

    for phage, host in flip_targets:
        if phage in parasite_idx and host in host_idx:
            i, j = parasite_idx[phage], host_idx[host]
            # flip_cost_matrix[i, :] = 50
            targeted_phages.add(phage)
            if mat_corrupt[i, j] == 1:
                mat_corrupt[i, j] = 0
                # flip_cost_matrix[i, j] = 0.0
                flipped_indices.append((i, j))

    # Set inf for all non-targeted phage rows
    for p in parasites:
        if p not in targeted_phages:
            i = parasite_idx[p]
            # flip_cost_matrix[i, :] = 1
            mat_corrupt[i, :] = 1

    return mat_corrupt, flipped_indices

In [ ]:

parasite_idx = {p: i for i, p in enumerate(parasites)}
host_idx = {h: j for j, h in enumerate(hosts)}
print (flip_cost_matrix[parasite_idx["k141_65862"],:])

In [126]:
#  count number of 1 s in the mat
print(np.sum(mat == 1))

66


In [138]:


flip_targets = [("k141_346380", "bin.99")]
# corrupt_mat, hidden, flip_cost_matrix = flip_specific_cells(mat, parasites, hosts, flip_targets, flip_cost_matrix)
corrupt_mat, hidden= flip_specific_cells(mat, parasites, hosts, flip_targets)

# find the entries of the flip cost matrix that are greater than 0
# print(np.count_nonzero(flip_cost_matrix > 0))

# Save hidden cells for metrics
hidden_cells = [(parasites[i], hosts[j]) for i, j in hidden]

# highlight_corrupted = {"corrupted": hidden_cells}
# plot_matrix(corrupt_mat, parasites, hosts,
#             filename=os.path.join(outdir, "corrupted_sample_cow.svg"),
#             highlight=highlight_corrupted)

# Update out["cell_state"] with corrupted values
corrupt_cell_state = {}
for i, p in enumerate(parasites):
    for j, h in enumerate(hosts):
        corrupt_cell_state[(p, h)] = int(corrupt_mat[i, j])
out["cell_state"] = corrupt_cell_state



lambda_param, cut_result = binary_search_lambda(
    out, parasites, hosts, hidden_cells=hidden_cells,
    target_flips=1,
    host_W_matrices=host_W_matrices, par_W_matrices=par_W_matrices,
    flip_cost_matrix=flip_cost_matrix,
    tol=0, max_iter=20
)

[iter 0] λ=0.5000 → flips=19
[iter 1] λ=0.2500 → flips=19
[iter 2] λ=0.1250 → flips=0
[iter 3] λ=0.1875 → flips=0
[iter 4] λ=0.2188 → flips=0
[iter 5] λ=0.2344 → flips=0
[iter 6] λ=0.2422 → flips=0
[iter 7] λ=0.2461 → flips=0
[iter 8] λ=0.2480 → flips=0
[iter 9] λ=0.2490 → flips=19
[iter 10] λ=0.2485 → flips=19
[iter 11] λ=0.2483 → flips=19
[iter 12] λ=0.2482 → flips=19
[iter 13] λ=0.2481 → flips=19
[iter 14] λ=0.2481 → flips=0
[iter 15] λ=0.2481 → flips=0
[iter 16] λ=0.2481 → flips=0
[iter 17] λ=0.2481 → flips=19
[iter 18] λ=0.2481 → flips=19
[iter 19] λ=0.2481 → flips=0

Best λ=0.1250 → flips=0, correct=0/1


In [139]:
import pandas as pd
import os
import numpy as np

# Load dataset
tsv_path = "cow_dataset_hic_gtdb.tsv"
df = pd.read_csv(tsv_path, sep="\t")

results = []  # collect outputs for all pairs

acc=0
total=0

# Loop through each (Phage, MAGs) pair
for idx, row in df.iterrows():
    phage = row["Phage"]
    mag = row["MAGs"]

    flip_targets = [(phage, mag)]
    print(f"\n🧩 Processing pair {idx+1}/{len(df)}: {phage} ↔ {mag}")

    # Step 1: flip specific cell
    corrupt_mat, hidden = flip_specific_cells(mat, parasites, hosts, flip_targets)

    # Step 2: record hidden cells
    hidden_cells = [(parasites[i], hosts[j]) for i, j in hidden]

    # Step 3: visualize corrupted matrix
    # highlight_corrupted = {"corrupted": hidden_cells}
    # out_svg = os.path.join(outdir, f"corrupted_{phage}_{mag}.svg")
    # plot_matrix(
    #     corrupt_mat,
    #     parasites,
    #     hosts,
    #     filename=out_svg,
    #     highlight=highlight_corrupted,
    # )

    # Step 4: update out["cell_state"]
    corrupt_cell_state = {
        (p, h): int(corrupt_mat[i, j])
        for i, p in enumerate(parasites)
        for j, h in enumerate(hosts)
    }
    out["cell_state"] = corrupt_cell_state

    # Step 5: run optimization
    lambda_param, cut_result = binary_search_lambda(
        out,
        parasites,
        hosts,
        hidden_cells=hidden_cells,
        target_flips=3,
        host_W_matrices=host_W_matrices,
        par_W_matrices=par_W_matrices,
        flip_cost_matrix=flip_cost_matrix,
        tol=0,
        max_iter=20,
    )

    new_mat = cut_result["new_matrix"]
    new_cell_state = cut_result["new_cell_state"]
    flips = cut_result["flips"]

    # highlight_flipped = {"flipped": [(p, h) for p, h, old, new in flips]}
    print(f"Number of flips performed: {len(flips)}")
    # for p, h, old, new in flips:
    #     print(f"Cell ({p},{h}): {old} -> {new}")

    # ---------------------------
    # Step 3: Save recovered matrix with algorithm flips
    # ---------------------------
    # plot_matrix(new_mat, parasites, hosts,
    #             filename=os.path.join(outdir, "flipped_matrix.svg"),
    #             highlight=highlight_flipped)






    # inside main(), after you have flips and hidden_cells
    metrics = compute_metrics(hidden_cells, flips)
    print("Metrics:", metrics)

    # Step 6: collect result
    results.append({
        "Phage": phage,
        "MAG": mag,
        "lambda": lambda_param,
        "metrics": metrics
        
    })

    print(f"✅ Finished {phage} - {mag}, λ={lambda_param}, metrics={metrics}")

    total+=1
    if metrics["recall"]==1:
        acc+=1
print(f"Accuracy: {acc}/{total} = {acc/total:.2f}")
# Optional: save summary
results_df = pd.DataFrame(results)
results_df.to_csv(os.path.join(outdir, "flip_experiment_summary.tsv"), sep="\t", index=False)

print("\n🎉 All pairs processed and results saved.")



🧩 Processing pair 1/66: k141_346380 ↔ bin.99
[iter 0] λ=0.5000 → flips=19
[iter 1] λ=0.2500 → flips=19
[iter 2] λ=0.1250 → flips=0
[iter 3] λ=0.1875 → flips=0
[iter 4] λ=0.2188 → flips=0
[iter 5] λ=0.2344 → flips=0
[iter 6] λ=0.2422 → flips=0
[iter 7] λ=0.2461 → flips=0
[iter 8] λ=0.2480 → flips=0
[iter 9] λ=0.2490 → flips=19
[iter 10] λ=0.2485 → flips=19
[iter 11] λ=0.2483 → flips=19
[iter 12] λ=0.2482 → flips=19
[iter 13] λ=0.2481 → flips=19
[iter 14] λ=0.2481 → flips=0
[iter 15] λ=0.2481 → flips=0
[iter 16] λ=0.2481 → flips=0
[iter 17] λ=0.2481 → flips=19
[iter 18] λ=0.2481 → flips=19
[iter 19] λ=0.2481 → flips=0

Best λ=0.1250 → flips=0, correct=0/3
Number of flips performed: 0
Metrics: {'precision': 0, 'recall': 0.0, 'f1': 0, 'TP': 0, 'FP': 0, 'FN': 1}
✅ Finished k141_346380 - bin.99, λ=0.125, metrics={'precision': 0, 'recall': 0.0, 'f1': 0, 'TP': 0, 'FP': 0, 'FN': 1}

🧩 Processing pair 2/66: k141_512174 ↔ bin.97
[iter 0] λ=0.5000 → flips=41
[iter 1] λ=0.2500 → flips=2
[iter 2] λ

In [18]:

cut_result = solve_network_cut(out, host_W_matrices=host_W_matrices, par_W_matrices=par_W_matrices,
                                         flip_cost_matrix=flip_cost_matrix, lambda_param=lambda_param)
# ---------------------------
# Step 2: Run network cut recovery on corrupted input
# ---------------------------


new_mat = cut_result["new_matrix"]
new_cell_state = cut_result["new_cell_state"]
flips = cut_result["flips"]

highlight_flipped = {"flipped": [(p, h) for p, h, old, new in flips]}
print(f"Number of flips performed: {len(flips)}")
# for p, h, old, new in flips:
#     print(f"Cell ({p},{h}): {old} -> {new}")

# ---------------------------
# Step 3: Save recovered matrix with algorithm flips
# ---------------------------
plot_matrix(new_mat, parasites, hosts,
            filename=os.path.join(outdir, "flipped_matrix_sample_cow.svg"),
            highlight=highlight_flipped)






# inside main(), after you have flips and hidden_cells
metrics = compute_metrics(hidden_cells, flips)
print("Metrics:", metrics)

Number of flips performed: 2
Matrix saved as experiments/flipped_matrix_sample_cow.svg
Metrics: {'precision': 0.5, 'recall': 1.0, 'f1': 0.6666666666666666, 'TP': 1, 'FP': 1, 'FN': 0}
